<a href="https://colab.research.google.com/github/vivomouallem12-oss/Cloud-Project/blob/main/Practice3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install firebase

In [ ]:
from firebase import firebase
import re

In [ ]:
FIREBASE_URL = "https://project1-23386-default-rtdb.firebaseio.com/"
FBconn = firebase.FirebaseApplication(FIREBASE_URL, None)

In [ ]:
def normalize_word(word):
    """
    Clean one single word:
    - lowercase
    - remove punctuation
    - keep letters/numbers
    """
    word = word.strip().lower()
    word = re.sub(r"[^\w\u0590-\u05FF\u0600-\u06FF]", "", word)
    return word


def extract_words(text):
    """
    Extract separate words from a full text.
    """
    text = text.lower()
    words = re.findall(r"[\w\u0590-\u05FF\u0600-\u06FF]+", text)
    return words


def get_all_words():
    data = FBconn.get('/words', None)
    return data if data else {}


def get_word_count(word):
    word = normalize_word(word)
    if not word:
        return None

    data = FBconn.get(f'/words/{word}', None)
    if data is None:
        return None

    return data.get("count", 0)


def add_single_word(word):
    """
    Add exactly one word only.
    If user enters more than one word -> reject it.
    """
    raw = word.strip()
    parts = raw.split()

    if len(parts) != 1:
        print("Error: option 1 accepts one word only.")
        return

    word = normalize_word(parts[0])

    if not word:
        print("Invalid word.")
        return

    current_count = get_word_count(word)

    if current_count is None:
        FBconn.put('/words', word, {"count": 1})
        print(f"Added new word: {word} -> 1")
    else:
        FBconn.put(f'/words/{word}', 'count', current_count + 1)
        print(f"Updated word: {word} -> {current_count + 1}")


def add_full_text(text):
    """
    Add all words from a full text.
    Example:
    'hello world hello' -> hello +2, world +1
    """
    words = extract_words(text)

    if not words:
        print("No valid words found.")
        return

    for word in words:
        current_count = get_word_count(word)

        if current_count is None:
            FBconn.put('/words', word, {"count": 1})
        else:
            FBconn.put(f'/words/{word}', 'count', current_count + 1)

    print("Finished analyzing full text.")


def update_word_count(word, new_count):
    word = normalize_word(word)

    if not word:
        print("Invalid word.")
        return

    if new_count < 0:
        print("Count cannot be negative.")
        return

    current_count = get_word_count(word)

    if current_count is None:
        print(f"Word '{word}' does not exist.")
        return

    FBconn.put(f'/words/{word}', 'count', new_count)
    print(f"Word '{word}' updated to {new_count}.")


def delete_word(word):
    word = normalize_word(word)

    if not word:
        print("Invalid word.")
        return

    current_count = get_word_count(word)

    if current_count is None:
        print(f"Word '{word}' does not exist.")
        return

    FBconn.delete('/words', word)
    print(f"Word '{word}' deleted.")


def show_all_words():
    data = get_all_words()

    if not data:
        print("No words found.")
        return

    print("\nSaved words:")
    for word in sorted(data.keys()):
        print(f"{word}: {data[word]['count']}")

In [ ]:
def menu():
    while True:
        print("\n===== WORD COUNTER MENU =====")
        print("1. Add one word")
        print("2. Add full text for analysis")
        print("3. Update word count")
        print("4. Delete a word")
        print("5. View all saved words")
        print("6. Exit")

        choice = input("Enter your choice (1-6): ").strip()

        if choice == '1':
            word = input("Enter one word: ")
            add_single_word(word)

        elif choice == '2':
            text = input("Enter full text: ")
            add_full_text(text)

        elif choice == '3':
            word = input("Enter the word to update: ")
            try:
                new_count = int(input("Enter the new count: "))
                update_word_count(word, new_count)
            except ValueError:
                print("Please enter a valid integer.")

        elif choice == '4':
            word = input("Enter the word to delete: ")
            delete_word(word)

        elif choice == '5':
            show_all_words()

        elif choice == '6':
            print("Goodbye.")
            break

        else:
            print("Invalid choice. Try again.")

In [ ]:
menu()


===== WORD COUNTER MENU =====
1. Add one word
2. Add full text for analysis
3. Update word count
4. Delete a word
5. View all saved words
6. Exit
Enter your choice (1-6): 1
Enter one word: hello
Updated word: hello -> 3

===== WORD COUNTER MENU =====
1. Add one word
2. Add full text for analysis
3. Update word count
4. Delete a word
5. View all saved words
6. Exit
Enter your choice (1-6): 2
Enter full text: hello world
Finished analyzing full text.

===== WORD COUNTER MENU =====
1. Add one word
2. Add full text for analysis
3. Update word count
4. Delete a word
5. View all saved words
6. Exit
Enter your choice (1-6): 5

Saved words:
hello: 4
helloworld: 1
world: 1

===== WORD COUNTER MENU =====
1. Add one word
2. Add full text for analysis
3. Update word count
4. Delete a word
5. View all saved words
6. Exit
Enter your choice (1-6): 6
Goodbye.
